In [41]:
import pandas as pd 
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [42]:
df = pd.read_csv(r'./datas/train.csv')
df.head()

C:\Users\test\AppData\Local\Temp\ipykernel_11836\1962537682.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'./datas/train.csv')


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [43]:
df.shape

(1017209, 9)

In [44]:
store = pd.read_csv(r'./datas/store.csv')

In [45]:
df = pd.merge(df, store, on='Store', how='inner')

In [46]:
df.isnull().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

In [47]:
df.loc[:,'CompetitionDistance'] = df['CompetitionDistance'].fillna(100000)
df.loc[:,'CompetitionOpenSinceMonth'] = df['CompetitionOpenSinceMonth'].fillna(0)
df.loc[:,'CompetitionOpenSinceYear'] = df['CompetitionOpenSinceYear'].fillna(0)
df.loc[:,'Promo2SinceWeek'] = df['Promo2SinceWeek'].fillna(0)
df.loc[:,'Promo2SinceYear'] = df['Promo2SinceYear'].fillna(0)
df.loc[:,'PromoInterval'] = df['PromoInterval'].fillna('None')

In [48]:
df['Date'] = pd.to_datetime(df['Date'])

In [49]:
all_stores = df['Store'].unique()
np.random.seed(42)
selected_stores = np.random.choice(all_stores, size=200, replace=False)
new_df = df.loc[df['Store'].isin(selected_stores)]
print(f'Total Rows: {len(new_df)}')
print(f'')
print(f'Sequence integrity: {new_df.groupby('Store')['Date'].diff().nunique()}')


Total Rows: 183063

Sequence integrity: 2


In [50]:
df  = new_df.copy()

In [51]:


df = df.sort_values(['Store', 'Date'], ascending=[True, True])
df['Year'] =  df['Date'].dt.year 
df['Month'] = df['Date'].dt.month 
df['Day']  = df['Date'].dt.day_of_week 
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)

df['Day_sin'] = np.sin(2 * np.pi * df['Day']/7)
df['Day_cos'] = np.cos(2 * np.pi * df['Day']/7)
df['Month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['Month_cos'] = np.cos(2 * np.pi* df['Month']/12)
     


In [52]:
df['StateHoliday'] = df['StateHoliday'].astype(str)
df['StateHoliday'] = df['StateHoliday'].map({'0': 0, 0:0, 'a': 1, 'b': 1, 'c': 1})

In [53]:
df['Open'] = df['Open'].astype(np.float32)

In [54]:

df['CompOpenDate'] = pd.to_datetime(dict(year=df.CompetitionOpenSinceYear.fillna(1900), month=df.CompetitionOpenSinceMonth.fillna(1), day=1), errors='coerce')
df['DaysWithComp'] = (df['Date'] - df['CompOpenDate']).dt.days 
df['MonthsWithComp'] = df['DaysWithComp'] // 30
df.loc[df.MonthsWithComp < 0, 'MonthsWithComp'] = 0 
df['MonthsWithComp'].fillna(0)
    


1016098    40.0
1014983    40.0
1013868    40.0
1012753    40.0
1011638    40.0
           ... 
5568       52.0
4453       52.0
3338       52.0
2223       52.0
1108       52.0
Name: MonthsWithComp, Length: 183063, dtype: float64

In [55]:
df['Sales_log'] = np.log1p(df['Sales'])

In [56]:
df.head().T

,1016098,1014983,1013868,1012753,1011638
Store,4,4,4,4,4
DayOfWeek,2,3,4,5,6
Date,2013-01-01 00:00:00,2013-01-02 00:00:00,2013-01-03 00:00:00,2013-01-04 00:00:00,2013-01-05 00:00:00
Sales,0,9941,8247,8290,10338
Customers,0,1429,1248,1232,1514
Open,0.0,1.0,1.0,1.0,1.0
Promo,0,0,0,0,0
StateHoliday,1,0,0,0,0
SchoolHoliday,1,1,1,1,0
StoreType,c,c,c,c,c


In [57]:
weight = 100 
sales_mean = df['Sales'].mean()
Store_sales_agg = df.groupby('Store')['Sales'].agg(['count', 'mean'])
store_means = (Store_sales_agg['count'] * Store_sales_agg['mean'] + weight * sales_mean) / (Store_sales_agg['count'] + weight)
df['Store_encoded'] = df['Store'].map(store_means)

In [ ]:
weight = 100
sales_mean = df['Sales'].mean()
Store_sales_agg = df.groupby('Store')['Sales'].agg(['count', 'mean'])
store_means = (store_sales_agg['count'] * store_sales_agg['mean'] + weight * sales_mean) / (store_sales_agg['count'] + weight)
df['store_encoded'] = df['Store'].map(store_means)

In [ ]:
import pickle 
with  open('store_map.pkl', 'wb') as f:
    pickle.dump(store_means, f)
print("store mapping saved successfully")

In [58]:
df.isnull().sum()

Store                            0
DayOfWeek                        0
Date                             0
Sales                            0
Customers                        0
Open                             0
Promo                            0
StateHoliday                     0
SchoolHoliday                    0
StoreType                        0
Assortment                       0
CompetitionDistance              0
CompetitionOpenSinceMonth        0
CompetitionOpenSinceYear         0
Promo2                           0
Promo2SinceWeek                  0
Promo2SinceYear                  0
PromoInterval                    0
Year                             0
Month                            0
Day                              0
WeekOfYear                       0
Day_sin                          0
Day_cos                          0
Month_sin                        0
Month_cos                        0
CompOpenDate                 48248
DaysWithComp                 48248
MonthsWithComp      

In [59]:
feature_cols = ['Day_sin', 'Day_cos', 'Month_sin', 'Store','Month_cos','Store_encoded', 'MonthsWithComp', 'Sales_log', 'StateHoliday', 'Open', 'DayOfWeek']
df = df[feature_cols]

In [60]:
df['MonthsWithComp'] = df['MonthsWithComp'].fillna(0)
df.fillna(0, inplace=True)
print(df.isnull().sum().sum())

0


In [61]:
df.to_csv(r'./datas/Rossman_feature_engineered.csv')